In [ ]:
pip install numpy pandas scipy xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.0 MB/s eta 0:00:00


In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG
# ============================================================

TOTAL_PERIODS = 20
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,13,17,20])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    for t in range(first_anchor, total_periods + 1):
        if t <= anchor_periods[0]:
            g[t] = anchor_rates[0]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    Generic MAU simulation:
      - Periods < start_period: MAU = 0
      - Period start_period: MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    g[t] is growth from t-1 to t.
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau

# ============================================================
# 2. COUNTRY CONFIGS (with your targets)
#    -> EVERY country "start_mode": "solve"
# ============================================================

country_configs = {
    "US": {
        "start_period": 1,
        "anchor_periods": np.array([2, 5, 9, 12], dtype=int),
        # US targets (Year 1–5) at periods 4,8,12,16,20
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.8991511,
            4.9817562,
            10.1063189
        ], dtype=float),
        # x0: [start_mau_guess, anchors...]
        "x0": np.array([0.5, 0.60, 0.45, 0.35, 0.20, 0.10, 0.05], dtype=float),
    },

    "Japan": {
        "start_period": 5,         # launch at period 5
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        # Japan targets at Years 2–5 → periods 8,12,16,20
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.5226359,
            1.0592352,
            1.3172527
        ], dtype=float),
        # x0: [start_mau_guess_at_period5, anchors...]
        "x0": np.array([0.05, 1.20, 0.51, 0.18, 0.09, 0.09], dtype=float),
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.
        ], dtype=float),
        "x0": np.array([0.05, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },

    "France": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.0780557179,
            0.4289106201,
            0.8605848599,
            1.0568479492
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.0520211969,
            0.2833763724,
            0.5620205078,
            0.6802569724
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },
}

# ============================================================
# 3. SOLVER PER COUNTRY (start_mau + anchors for each)
# ============================================================

def solve_country(name, cfg):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = cfg["x0"]

    # x = [start_mau, anchor_rates...]
    def residuals(x):
        start_mau    = x[0]
        anchor_rates = x[1:]
        g   = build_growth_series(anchor_periods, anchor_rates)
        mau = simulate_maus(start_mau, start_period, g)
        return mau[target_periods] - target_maus

    lower_bounds = np.zeros_like(x0)
    upper_bounds = np.full_like(x0, np.inf)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=30000,
        xtol=1e-12,
        ftol=1e-12
    )
    x_opt       = result.x
    start_mau   = x_opt[0]
    anchor_rates = x_opt[1:]

    g_full   = build_growth_series(anchor_periods, anchor_rates)
    mau_full = simulate_maus(start_mau, start_period, g_full)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,   # decimals per period
        "mau_series":    mau_full, # MAUs per period
    }

# ============================================================
# 4. RUN SOLVER FOR ALL COUNTRIES
# ============================================================

results = {}

for country_name, cfg in country_configs.items():
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    print("  Anchor growth rates (%):")
    for p, r in zip(cfg["anchor_periods"], res["anchor_rates"]):
        print(f"    Period {p}: {r*100:.6f}%")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        print(f"    Period {tp}: simulated={mau[tp]:.6f}, target={target:.6f}")

# ============================================================
# 5. BUILD EXCEL OUTPUT
# ============================================================

# Sheet 1: Growth percentages per period
growth_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    g = res["growth_series"]
    growth_df[country_name + "_growth_pct"] = [g[t] * 100 for t in PERIODS]

# Sheet 2: MAUs per period
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    m = res["mau_series"]
    mau_df[country_name + "_MAU_M"] = [m[t] for t in PERIODS]

# Sheet 3: Starting Acquisition Rates table (anchors only)
rows = []
for country_name, cfg in country_configs.items():
    anchor_periods = cfg["anchor_periods"]
    anchor_rates   = results[country_name]["anchor_rates"]
    for p, r in zip(anchor_periods, anchor_rates):
        rows.append({
            "Country": country_name,
            "Period": p,
            "Starting Acquisition Rate (High Case)": r * 100
        })

anchor_df = pd.DataFrame(rows).sort_values(["Country", "Period"])

output_filename = "growth_model_all_countries_independent.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs", index=False)
    anchor_df.to_excel(writer, sheet_name="Starting_Acquisition_Rates", index=False)

print(f"\nExcel file written to: {output_filename}")


# Five Year

In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG
# ============================================================

TOTAL_PERIODS = 20
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

# strength of the penalty for anchor increases
MONO_LAMBDA = 50.0  # you can tune this (e.g. 10, 50, 100)


def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,13,17,20])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    for t in range(first_anchor, total_periods + 1):
        if t <= anchor_periods[0]:
            g[t] = anchor_rates[0]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    Generic MAU simulation:
      - Periods < start_period: MAU = 0
      - Period start_period: MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    g[t] is growth from t-1 to t.
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau


def monotonic_penalty(anchor_rates, lam=MONO_LAMBDA):
    """
    Penalize any increases in anchor_rates.
    We want anchor_rates to be non-increasing:
        anchor[i+1] <= anchor[i]
    For each violation v = max(0, anchor[i+1] - anchor[i]),
    we add a residual term sqrt(lam) * v, so squared error = lam * v^2.
    """
    anchor_rates = np.asarray(anchor_rates, float)
    diffs = anchor_rates[1:] - anchor_rates[:-1]  # positive if increasing
    violations = np.clip(diffs, 0, np.inf)        # keep only increases
    # return vector of residuals, one per violation
    return np.sqrt(lam) * violations


# ============================================================
# 2. COUNTRY CONFIGS (with your targets)
#    -> EVERY country solves its own start_mau + anchors
# ============================================================

country_configs = {
    "US": {
        "start_period": 1,
        "anchor_periods": np.array([2, 5, 9, 13, 17, 20], dtype=int),
        # US targets (Year 1–5) at periods 4,8,12,16,20
        "target_periods": np.array([4, 8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.7519024,
            4.1659231,
            8.4512663,
            9.9756940,
            10.5096795
        ], dtype=float),
        # x0: [start_mau_guess, anchors...]
        "x0": np.array([0.5, 0.60, 0.45, 0.35, 0.20, 0.10, 0.05], dtype=float),
    },

    "Japan": {
        "start_period": 5,         # launch at period 5
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        # Japan targets at Years 2–5 → periods 8,12,16,20
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.5599089,
            1.0969580,
            1.3075732,
            1.3308565
        ], dtype=float),
        # x0: [start_mau_guess_at_period5, anchors...]
        "x0": np.array([0.05, 1.20, 0.51, 0.18, 0.09, 0.09], dtype=float),
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.4923796,
            0.9979143,
            1.2409946,
            1.3289723
        ], dtype=float),
        "x0": np.array([0.05, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },

    "France": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.4310815,
            0.8649407,
            1.0621972,
            1.1204738
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 13, 17, 20], dtype=int),
        "target_periods": np.array([8, 12, 16, 20], dtype=int),
        "target_maus": np.array([
            0.4134837,
            0.8200624,
            0.9925850,
            1.0289795
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28, 0.17, 0.08], dtype=float),
    },
}

# ============================================================
# 3. SOLVER PER COUNTRY (start_mau + anchors + monotonic penalty)
# ============================================================

def solve_country(name, cfg):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = cfg["x0"]

    # x = [start_mau, anchor_rates...]
    def residuals(x):
        start_mau    = x[0]
        anchor_rates = x[1:]

        # main MAU fit residuals
        g   = build_growth_series(anchor_periods, anchor_rates)
        mau = simulate_maus(start_mau, start_period, g)
        mau_resids = mau[target_periods] - target_maus  # shape (n_targets,)

        # monotonicity penalty on anchors: penalize any increases
        mono_resids = monotonic_penalty(anchor_rates, lam=MONO_LAMBDA)  # shape (n_anchors-1,)

        # concatenate: least_squares will minimize sum of squares of all
        return np.concatenate([mau_resids, mono_resids])

    lower_bounds = np.zeros_like(x0)
    upper_bounds = np.full_like(x0, np.inf)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=30000,
        xtol=1e-12,
        ftol=1e-12
    )
    x_opt        = result.x
    start_mau    = x_opt[0]
    anchor_rates = x_opt[1:]

    g_full   = build_growth_series(anchor_periods, anchor_rates)
    mau_full = simulate_maus(start_mau, start_period, g_full)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,   # decimals per period
        "mau_series":    mau_full, # MAUs per period
    }

# ============================================================
# 4. RUN SOLVER FOR ALL COUNTRIES
# ============================================================

results = {}

for country_name, cfg in country_configs.items():
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    print("  Anchor growth rates (%):")
    for p, r in zip(cfg["anchor_periods"], res["anchor_rates"]):
        print(f"    Period {p}: {r*100:.6f}%")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        print(f"    Period {tp}: simulated={mau[tp]:.6f}, target={target:.6f}")

# ============================================================
# 5. BUILD EXCEL OUTPUT (Final formatting with full precision)
# ============================================================

# Sheet 1: Growth percentages per period (percent values, not decimals)
growth_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    g = res["growth_series"]
    # convert decimal → percent (e.g., 0.452 → 45.2)
    growth_df[country_name + "_growth_pct"] = [g[t] * 100.0 for t in PERIODS]

# Sheet 2: MAUs per period — ABSOLUTE NUMBERS WITH FULL DECIMALS
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    m = res["mau_series"]
    # multiply millions → exact subscriber count (float)
    mau_df[country_name + "_MAU_absolute"] = [m[t] * 1_000_000.0 for t in PERIODS]

# Sheet 3: Starting Acquisition Rates — pure decimals (e.g., 0.452)
rows = []
for country_name, cfg in country_configs.items():
    anchor_periods = cfg["anchor_periods"]
    anchor_rates   = results[country_name]["anchor_rates"]

    for p, r in zip(anchor_periods, anchor_rates):
        rows.append({
            "Country": country_name,
            "Period": p,
            "Starting Acquisition Rate (decimal)": r  # DO NOT convert to percent
        })

anchor_df = pd.DataFrame(rows).sort_values(["Country", "Period"])

output_filename = "growth_model_all_countries_monotonic.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs_Absolute", index=False)
    anchor_df.to_excel(writer, sheet_name="Starting_Acq_Rates", index=False)

print(f"\nExcel file written to: {output_filename}")




=== Solving for US ===
  Start MAU (Period 1): 0.154937 M
  Anchor growth rates (%):
    Period 2: 69.305230%
    Period 5: 69.302949%
    Period 9: 28.106318%
    Period 13: 5.199708%
    Period 17: 2.628487%
    Period 20: 0.005273%
    Period 4: simulated=0.751902, target=0.751902
    Period 8: simulated=4.165923, target=4.165923
    Period 12: simulated=8.451266, target=8.451266
    Period 16: simulated=9.975694, target=9.975694
    Period 20: simulated=10.509680, target=10.509680

=== Solving for Japan ===
  Start MAU (Period 5): 0.051574 M
  Anchor growth rates (%):
    Period 6: 175.115816%
    Period 9: 25.307922%
    Period 13: 6.941444%
    Period 17: 0.442690%
    Period 20: 0.441751%
    Period 8: simulated=0.559909, target=0.559909
    Period 12: simulated=1.096958, target=1.096958
    Period 16: simulated=1.307573, target=1.307573
    Period 20: simulated=1.330856, target=1.330857

=== Solving for UK ===
  Start MAU (Period 5): 0.206409 M
  Anchor growth rates (%):
    P

In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG  (NOW ONLY TO PERIOD 12)
# ============================================================

TOTAL_PERIODS = 12
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

# strength of the penalty for anchor increases
MONO_LAMBDA = 50.0  # you can tune this (e.g. 10, 50, 100)


def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,12])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    for t in range(first_anchor, total_periods + 1):
        if t <= anchor_periods[0]:
            g[t] = anchor_rates[0]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    Generic MAU simulation:
      - Periods < start_period: MAU = 0
      - Period start_period: MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    g[t] is growth from t-1 to t.
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau


def monotonic_penalty(anchor_rates, lam=MONO_LAMBDA):
    """
    Penalize any increases in anchor_rates.
    We want anchor_rates to be non-increasing:
        anchor[i+1] <= anchor[i]
    For each violation v = max(0, anchor[i+1] - anchor[i]),
    we add a residual term sqrt(lam) * v, so squared error = lam * v^2.
    """
    anchor_rates = np.asarray(anchor_rates, float)
    diffs = anchor_rates[1:] - anchor_rates[:-1]  # positive if increasing
    violations = np.clip(diffs, 0, np.inf)        # keep only increases
    return np.sqrt(lam) * violations


# ============================================================
# 2. COUNTRY CONFIGS (targets truncated to <= period 12)
#    -> EVERY country solves its own start_mau + anchors
# ============================================================

country_configs = {
    "US": {
        "start_period": 1,
        # only anchors up to 12
        "anchor_periods": np.array([2, 5, 9, 12], dtype=int),
        # US targets now only at periods 4, 8, 12 (first 3 values)
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.899151,
            4.981756,
            10.106319
        ], dtype=float),
        # x0: [start_mau_guess, anchors...]
        "x0": np.array([0.5, 0.60, 0.45, 0.35, 0.20], dtype=float),
    },

    "Japan": {
        "start_period": 5,         # launch at period 5
        # only anchors up to 12
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        # Japan targets now only at periods 8, 12 (first 2 values)
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.604164,
            1.183661
        ], dtype=float),
        # x0: [start_mau_guess_at_period5, anchors...]
        "x0": np.array([0.05, 1.20, 0.51, 0.18], dtype=float),
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.522636,
            1.059235
        ], dtype=float),
        "x0": np.array([0.05, 0.37, 0.35, 0.28], dtype=float),
    },

    "France": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.457571,
            0.918091
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": np.array([6, 9, 12], dtype=int),
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.441795,
            0.876213
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },
}

# ============================================================
# 3. SOLVER PER COUNTRY (start_mau + anchors + monotonic penalty)
# ============================================================

def solve_country(name, cfg):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = cfg["x0"]

    # x = [start_mau, anchor_rates...]
    def residuals(x):
        start_mau    = x[0]
        anchor_rates = x[1:]

        # main MAU fit residuals
        g   = build_growth_series(anchor_periods, anchor_rates)
        mau = simulate_maus(start_mau, start_period, g)
        mau_resids = mau[target_periods] - target_maus  # shape (n_targets,)

        # monotonicity penalty on anchors: penalize any increases
        mono_resids = monotonic_penalty(anchor_rates, lam=MONO_LAMBDA)  # shape (n_anchors-1,)

        # concatenate: least_squares will minimize sum of squares of all
        return np.concatenate([mau_resids, mono_resids])

    lower_bounds = np.zeros_like(x0)
    upper_bounds = np.full_like(x0, np.inf)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=30000,
        xtol=1e-12,
        ftol=1e-12
    )
    x_opt        = result.x
    start_mau    = x_opt[0]
    anchor_rates = x_opt[1:]

    g_full   = build_growth_series(anchor_periods, anchor_rates)
    mau_full = simulate_maus(start_mau, start_period, g_full)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,   # decimals per period
        "mau_series":    mau_full, # MAUs per period
    }

# ============================================================
# 4. RUN SOLVER FOR ALL COUNTRIES
# ============================================================

results = {}

for country_name, cfg in country_configs.items():
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    print("  Anchor growth rates (%):")
    for p, r in zip(cfg["anchor_periods"], res["anchor_rates"]):
        print(f"    Period {p}: {r*100:.6f}%")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        print(f"    Period {tp}: simulated={mau[tp]:.6f}, target={target:.6f}")

# ============================================================
# 5. BUILD EXCEL OUTPUT (Final formatting with full precision)
# ============================================================

# Sheet 1: Growth percentages per period (percent values, not decimals)
growth_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    g = res["growth_series"]
    # convert decimal → percent (e.g., 0.452 → 45.2)
    growth_df[country_name + "_growth_pct"] = [g[t] * 100.0 for t in PERIODS]

# Sheet 2: MAUs per period — ABSOLUTE NUMBERS WITH FULL DECIMALS
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    m = res["mau_series"]
    mau_df[country_name + "_MAU_absolute"] = [m[t] * 1_000_000.0 for t in PERIODS]

# Sheet 3: Starting Acquisition Rates — pure decimals (e.g., 0.452)
rows = []
for country_name, cfg in country_configs.items():
    anchor_periods = cfg["anchor_periods"]
    anchor_rates   = results[country_name]["anchor_rates"]

    for p, r in zip(anchor_periods, anchor_rates):
        rows.append({
            "Country": country_name,
            "Period": p,
            "Starting Acquisition Rate (decimal)": r
        })

anchor_df = pd.DataFrame(rows).sort_values(["Country", "Period"])

output_filename = "growth_model_all_countries_monotonic_P12.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs_Absolute", index=False)
    anchor_df.to_excel(writer, sheet_name="Starting_Acq_Rates", index=False)

print(f"\nExcel file written to: {output_filename}")



In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG  (UP TO PERIOD 12)
# ============================================================

TOTAL_PERIODS = 12
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

# Set to 0.0 to effectively disable monotonic penalty
MONO_LAMBDA = 0.0  # you can later increase if you want smoother anchors


def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,12])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    for t in range(first_anchor, total_periods + 1):
        if t <= anchor_periods[0]:
            g[t] = anchor_rates[0]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    Generic MAU simulation:
      - Periods < start_period: MAU = 0
      - Period start_period: MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    g[t] is growth from t-1 to t.
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau


def monotonic_penalty(anchor_rates, lam=MONO_LAMBDA):
    """
    Penalize any increases in anchor_rates.
    If lam = 0.0, this returns all zeros and has no effect on the fit.
    """
    if lam <= 0:
        return np.zeros(max(len(anchor_rates) - 1, 0), dtype=float)

    anchor_rates = np.asarray(anchor_rates, float)
    diffs = anchor_rates[1:] - anchor_rates[:-1]  # positive if increasing
    violations = np.clip(diffs, 0, np.inf)        # keep only increases
    return np.sqrt(lam) * violations


# ============================================================
# 2. COUNTRY CONFIGS
#    US → 3 years (targets at 4,8,12) with 4 anchors
#    Others → 2 years (targets at 8,12) with 3 anchors
# ============================================================

country_configs = {
    "US": {
        "start_period": 1,
        "anchor_periods": np.array([2, 5, 9, 12], dtype=int),
        # US targets (Year 1–3) at periods 4,8,12
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.8991511,
            4.9817562,
            10.1063189
        ], dtype=float),
        # x0: [start_mau_guess, 4 anchor guesses]
        "x0": np.array([0.5, 0.60, 0.45, 0.35, 0.20], dtype=float),
    },

    "Japan": {
        "start_period": 5,                         # launch at period 5
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # 2-year horizon from launch → periods 8, 12
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.60416370,
            1.18366086
        ], dtype=float),
        # x0: [start_mau_guess_at_period5, 3 anchor guesses]
        "x0": np.array([0.05, 1.20, 0.51, 0.18], dtype=float),
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # 2-year horizon from launch → 8,12
        "target_periods": np.array([8, 12], dtype=int),
        # TODO: replace with actual UK MAU targets
        "target_maus": np.array([
            0.52263589,
            1.05923524
        ], dtype=float),
        "x0": np.array([0.05, 0.37, 0.35, 0.28], dtype=float),
    },

    "France": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # Must be ≤ 12 because TOTAL_PERIODS = 12
        "target_periods": np.array([8, 12], dtype=int),
        # First two of your original targets (for 8,12)
        "target_maus": np.array([
            0.45757114,
            0.91809057
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # Must be ≤ 12 because TOTAL_PERIODS = 12
        "target_periods": np.array([8, 12], dtype=int),
        # First two of your original targets (for 8,12)
        "target_maus": np.array([
            0.44179525,
            0.87621275
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },
}


# ============================================================
# 3. SOLVER PER COUNTRY (start_mau + anchors + optional mono penalty)
# ============================================================

def solve_country(name, cfg):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = cfg["x0"]

    # x = [start_mau, anchor_rates...]
    def residuals(x):
        start_mau    = x[0]
        anchor_rates = x[1:]

        # main MAU fit residuals
        g   = build_growth_series(anchor_periods, anchor_rates)
        mau = simulate_maus(start_mau, start_period, g)
        mau_resids = mau[target_periods] - target_maus  # shape (n_targets,)

        # monotonicity penalty on anchors (zero if MONO_LAMBDA == 0)
        mono_resids = monotonic_penalty(anchor_rates, lam=MONO_LAMBDA)

        # concatenate: least_squares will minimize sum of squares of all
        return np.concatenate([mau_resids, mono_resids])

    lower_bounds = np.zeros_like(x0)
    upper_bounds = np.full_like(x0, np.inf)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=30000,
        xtol=1e-12,
        ftol=1e-12
    )
    x_opt        = result.x
    start_mau    = x_opt[0]
    anchor_rates = x_opt[1:]

    g_full   = build_growth_series(anchor_periods, anchor_rates)
    mau_full = simulate_maus(start_mau, start_period, g_full)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,   # decimals per period
        "mau_series":    mau_full, # MAUs per period
    }

# ============================================================
# 4. RUN SOLVER FOR ALL COUNTRIES
# ============================================================

results = {}

for country_name, cfg in country_configs.items():
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    print("  Anchor growth rates (%):")
    for p, r in zip(cfg["anchor_periods"], res["anchor_rates"]):
        print(f"    Period {p}: {r*100:.6f}%")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        print(f"    Period {tp}: simulated={mau[tp]:.6f}, target={target:.6f}")

# ============================================================
# 5. BUILD EXCEL OUTPUT (anchors only for growth; full timeline for MAUs)
# ============================================================

# Sheet 1: Growth percentages per anchor (4 for US, 3 for others)
growth_rows = []
for country_name, cfg in country_configs.items():
    anchor_periods = cfg["anchor_periods"]
    anchor_rates   = results[country_name]["anchor_rates"]

    for p, r in zip(anchor_periods, anchor_rates):
        growth_rows.append({
            "Country": country_name,
            "Period": p,
            "Growth_pct": r * 100.0
        })

growth_df = pd.DataFrame(growth_rows).sort_values(["Country", "Period"])

# Sheet 2: MAUs per period — ABSOLUTE NUMBERS WITH FULL DECIMALS
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name, res in results.items():
    m = res["mau_series"]
    mau_df[country_name + "_MAU_absolute"] = [m[t] * 1_000_000.0 for t in PERIODS]

# Sheet 3: Starting Acquisition Rates — pure decimals (e.g., 0.452)
anchor_rows = []
for country_name, cfg in country_configs.items():
    anchor_periods = cfg["anchor_periods"]
    anchor_rates   = results[country_name]["anchor_rates"]

    for p, r in zip(anchor_periods, anchor_rates):
        anchor_rows.append({
            "Country": country_name,
            "Period": p,
            "Starting Acquisition Rate (decimal)": r
        })

anchor_df = pd.DataFrame(anchor_rows).sort_values(["Country", "Period"])

output_filename = "growth_model_all_countries_fit_exact_P12_US3yrs_others2yrs.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs_Absolute", index=False)
    anchor_df.to_excel(writer, sheet_name="Starting_Acq_Rates", index=False)

print(f"\nExcel file written to: {output_filename}")


# Year 3


In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG  (UP TO PERIOD 12)
# ============================================================

TOTAL_PERIODS = 12
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

# Penalty strengths
MONO_LAMBDA    = 0.2   # small monotonic penalty on anchors (0 = off)
SMOOTH_LAMBDA  = 0.5   # smoothness penalty on anchor curvature

# Desired output order
country_order = ["US", "UK", "France", "Germany", "Japan"]


def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,12])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    for t in range(first_anchor, total_periods + 1):
        if t <= first_anchor:
            g[t] = anchor_rates[0]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    MAU simulation:
      - Periods < start_period: MAU = 0
      - Period start_period:    MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau


def monotonic_penalty(anchor_rates, lam=MONO_LAMBDA):
    """
    Penalize any increases in anchor_rates (we want them non-increasing).
    If lam = 0.0, returns all zeros and has no effect.
    """
    if lam <= 0:
        return np.zeros(max(len(anchor_rates) - 1, 0), dtype=float)

    anchor_rates = np.asarray(anchor_rates, float)
    diffs = anchor_rates[1:] - anchor_rates[:-1]  # positive if increasing
    violations = np.clip(diffs, 0, np.inf)        # keep only increases
    return np.sqrt(lam) * violations


def smoothness_penalty(anchor_rates, lam=SMOOTH_LAMBDA):
    """
    Penalize curvature (second differences) so anchor rates change smoothly.
    This encourages nice S-ish / exponential-ish curves without templates.
    """
    if lam <= 0:
        return np.zeros(len(anchor_rates), dtype=float)

    anchor_rates = np.asarray(anchor_rates, float)
    if len(anchor_rates) < 3:
        return np.zeros(len(anchor_rates), dtype=float)

    # second differences: r_{i+1} - 2 r_i + r_{i-1}
    second_diff = anchor_rates[2:] - 2 * anchor_rates[1:-1] + anchor_rates[:-2]
    return np.sqrt(lam) * second_diff


# ============================================================
# 2. COUNTRY CONFIGS
#    US → 3 years (targets at 4,8,12) with 4 anchors
#    Others → 2 years (targets at 8,12) with 3 anchors
# ============================================================

country_configs = {
    "US": {
        "start_period": 1,
        "anchor_periods": np.array([2, 5, 9, 12], dtype=int),
        # US targets (Year 1–3) at periods 4,8,12
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.7749768,
            4.2678931,
            8.5744581
        ], dtype=float),
        # x0: [start_mau_guess, 4 anchor guesses]
        "x0": np.array([0.5, 0.60, 0.45, 0.35, 0.30], dtype=float),
    },

    "Japan": {
        "start_period": 5,                         # launch at period 5
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # 2-year horizon from launch → periods 8, 12
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.6226316,
            1.2373884
        ], dtype=float),
        # x0: [start_mau_guess_at_period5, 3 anchor guesses]
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        # 2-year horizon from launch → 8,12
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.4487472,
            0.9015061
        ], dtype=float),
        "x0": np.array([0.05, 0.37, 0.35, 0.28], dtype=float),
    },

    "France": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.2809596,
            0.5625267
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": np.array([5, 9, 12], dtype=int),
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.3850503,
            0.7681255
        ], dtype=float),
        "x0": np.array([0.03, 0.37, 0.35, 0.28], dtype=float),
    },
}


# ============================================================
# 3. SOLVER PER COUNTRY (start_mau + anchors + smoothness)
# ============================================================

def solve_country(name, cfg):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = cfg["x0"]

    # x = [start_mau, anchor_rates...]
    def residuals(x):
        start_mau    = x[0]
        anchor_rates = x[1:]

        g   = build_growth_series(anchor_periods, anchor_rates)
        mau = simulate_maus(start_mau, start_period, g)
        mau_resids = mau[target_periods] - target_maus

        mono_resids   = monotonic_penalty(anchor_rates, lam=MONO_LAMBDA)
        smooth_resids = smoothness_penalty(anchor_rates, lam=SMOOTH_LAMBDA)

        return np.concatenate([mau_resids, mono_resids, smooth_resids])

    lower_bounds = np.zeros_like(x0)
    upper_bounds = np.full_like(x0, np.inf)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=30000,
        xtol=1e-12,
        ftol=1e-12
    )
    x_opt        = result.x
    start_mau    = x_opt[0]
    anchor_rates = x_opt[1:]

    g_full   = build_growth_series(anchor_periods, anchor_rates)
    mau_full = simulate_maus(start_mau, start_period, g_full)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,   # decimals per period
        "mau_series":    mau_full, # MAUs per period
    }


# ============================================================
# 4. RUN SOLVER FOR ALL COUNTRIES (IN DESIRED ORDER)
# ============================================================

results = {}

for country_name in country_order:
    cfg = country_configs[country_name]
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    print("  Anchor growth rates (%):")
    for p, r in zip(cfg["anchor_periods"], res["anchor_rates"]):
        print(f"    Period {p}: {r*100:.6f}%")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        print(f"    Period {tp}: simulated={mau[tp]:.6f}, target={target:.6f}")


# ============================================================
# 5. BUILD EXCEL OUTPUT
#    - Sheet 1: per-period growth in %
#        * US: periods 2–12
#        * Others: periods 6–12
#    - Sheet 2: MAUs per period (absolute)
#    - Sheet 3: per-period starting acquisition rates (decimals)
#        * US: periods 2–12
#        * Others: periods 6–12
# ============================================================

# Sheet 1: Growth percentages per period
growth_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    res = results[country_name]
    g = res["growth_series"]  # g[t] is decimal growth from t-1 → t

    col_pct = []
    for t in PERIODS:
        if country_name == "US":
            # US: report growth for 2–12
            if t >= 2:
                col_pct.append(g[t] * 100.0)
            else:
                col_pct.append(np.nan)
        else:
            # Others: report growth for 6–12
            if t >= 6:
                col_pct.append(g[t] * 100.0)
            else:
                col_pct.append(np.nan)

    growth_df[country_name + "_growth_pct"] = col_pct

# Sheet 2: MAUs per period — ABSOLUTE NUMBERS WITH FULL DECIMALS
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    res = results[country_name]
    m = res["mau_series"]
    mau_df[country_name + "_MAU_absolute"] = [m[t] * 1_000_000.0 for t in PERIODS]

# Sheet 3: Starting Acquisition Rates per period — DECIMALS
acq_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    res = results[country_name]
    g = res["growth_series"]

    col_acq = []
    for t in PERIODS:
        if country_name == "US":
            # US: periods 2–12
            if t >= 2:
                col_acq.append(g[t])
            else:
                col_acq.append(np.nan)
        else:
            # Others: periods 6–12
            if t >= 6:
                col_acq.append(g[t])
            else:
                col_acq.append(np.nan)

    acq_df[country_name + "_acq_rate"] = col_acq

output_filename = "growth_model_smooth_no_templates_P12_US_UK_FR_DE_JP.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs_Absolute", index=False)
    acq_df.to_excel(writer, sheet_name="Starting_Acq_Rates", index=False)

print(f"\nExcel file written to: {output_filename}")



=== Solving for US ===
  Start MAU (Period 1): 0.123254 M
  Anchor growth rates (%):
    Period 2: 94.778343%
    Period 5: 64.700288%
    Period 9: 34.622233%
    Period 12: 4.544178%
    Period 4: simulated=0.774977, target=0.774977
    Period 8: simulated=4.267893, target=4.267893
    Period 12: simulated=8.574458, target=8.574458

=== Solving for UK ===
  Start MAU (Period 5): 0.218264 M
  Anchor growth rates (%):
    Period 5: 31.205523%
    Period 9: 23.129459%
    Period 12: 15.053394%
    Period 8: simulated=0.448747, target=0.448747
    Period 12: simulated=0.901506, target=0.901506

=== Solving for France ===
  Start MAU (Period 5): 0.085914 M
  Anchor growth rates (%):
    Period 5: 63.102366%
    Period 9: 33.999189%
    Period 12: 4.896012%
    Period 8: simulated=0.280960, target=0.280960
    Period 12: simulated=0.562527, target=0.562527

=== Solving for Germany ===
  Start MAU (Period 5): 0.157361 M
  Anchor growth rates (%):
    Period 5: 42.693105%
    Period 9: 26.8

In [ ]:
|

# Ten Year

Extends the model from 12 periods (3 years) to **40 periods (10 years)** and
shapes it as an S-curve: slow launch, steep middle, flattening top.

**Saturation comes from the UMG2 Video P&L model** (v135MK, sheet
"MAUs - End of Year Targets"), not from an assumption. Section 2 replicates
that workbook's MAU build — `SOM x %Aware x %Converted`, where `%Aware` is a
logistic for three years and then the workbook's own decaying-growth
extrapolation — and runs the same recursion out to 2036. The replication is
asserted against the workbook's 2032 cells before anything depends on it.

That workbook implies a Year-10 level of roughly **2.0-2.2x the Year-3 level**
for every country. `CEILING_BASIS` picks how to apply it: `"multiple"`
(default) keeps this notebook's committed Y1-Y3 targets and applies that
multiple; `"absolute"` uses the workbook's own 2036 figure instead. The two
agree closely for UK/France/Germany/Japan and differ for the US, where this
notebook's Year-3 target runs ~27% above the workbook's 2029.

Other changes versus the Year 3 cell:

* `TOTAL_PERIODS = 40`, with growth anchors out to period 40 — dense through
  the ramp years, sparse across the plateau.
* A terminal growth target (`TERMINAL_GROWTH`) pins the last anchor near zero,
  which is what makes the top of the S actually flat instead of merely slower.
* Unknowns are solved in **log space**, and the smoothness penalty is applied to
  the log of the growth rate rather than the rate itself. Penalizing curvature
  of the raw rate forces a straight-line decay, which cannot both hold the
  Year-3 target and land on the Year-10 level — the solver answers that by
  crashing growth mid-horizon and re-accelerating, giving a double hump instead
  of an S. In log space a constant-percentage decay is curvature-free, so a
  clean monotone S is the natural answer. This also matches how the workbook
  itself extrapolates awareness.
* Target residuals are **relative**, not absolute, so the 0.77M Year-1 target
  carries the same weight as the Year-10 one.
* Optional quarterly seasonality (`USE_SEASONALITY`), normalized to a geometric
  mean of 1 and fit *through*, so the Y1-Y3 targets are still hit exactly.
  Off by default.

The Year 1-3 targets (periods 4, 8, 12) are unchanged and still hit to within
0.1%.

Outside the US the targets sit a year ahead of the workbook — periods 8 and 12
track its 2029 and 2030, while with launch at period 5 those periods are
calendar 2028 and 2029. That lead is intentional (the non-US plan is more
aggressive than the workbook), so the targets are taken at face value. See the
note on `pl_year_three`.

Section 6 prints how far `CEILING_BASIS` moves the path. Years 1-3 are pinned
by the targets either way, so the two bases can only differ from Year 4 on:
the US moves 24.7% and Japan 17.1% at Year 10, while UK, France and Germany
move about 1%.


In [ ]:
# @title
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# ============================================================
# 1. GLOBAL MODEL CONFIG  (10 YEARS = 40 QUARTERLY PERIODS)
# ============================================================

TOTAL_PERIODS = 40                       # 10 years x 4 quarters
PERIODS = np.arange(1, TOTAL_PERIODS + 1)

# Penalty strengths
MONO_LAMBDA    = 30.0   # push anchor growth rates to be non-increasing (decay)
SMOOTH_LAMBDA  = 3.0    # penalize curvature of the LOG rate, so decay stays geometric

# Residual weights (relative errors, so early small targets are not drowned out)
W_TARGET     = 100.0    # Year 1-3 commitments: hit these
W_CEILING    = 8.0      # Year 10 plateau: a planning assumption, softer
W_TERMINAL_G = 20.0     # terminal growth must land near zero (flat top)

# Shape of the plateau
MAX_GROWTH        = 1.50    # cap any single quarter at +150% (keeps the launch sane)
TERMINAL_GROWTH   = 0.002   # ~0.2% / quarter at period 40 -> curve is visibly flat

# Where the Year-10 level comes from. Both options are derived from the
# UMG2 Video P&L model (v135MK, sheet "MAUs - End of Year Targets"), which is
# replicated in section 2 below.
#   "multiple" - keep this notebook's committed Y1-Y3 targets and grow them by
#                the terminal multiple the P&L model implies (~2.0-2.2x the
#                Year-3 level). Respects the targets already in the notebook.
#   "absolute" - use the P&L model's own 2036 MAU outright. Use this if the
#                workbook, not the notebook, is the source of truth for levels.
# The two agree closely for UK/France/Germany/Japan; they differ for the US,
# where this notebook's Year-3 target runs ~27% above the workbook's 2029.
CEILING_BASIS     = "multiple"

# Optional quarterly seasonality (the wobble on the sketch).
# Values are multipliers on reported MAU for Q1..Q4; they are normalized to a
# geometric mean of 1 so they add wobble without changing the underlying trend.
# The solver fits THROUGH seasonality, so the Y1-Y3 targets are still hit exactly.
USE_SEASONALITY   = False
SEASONAL_FACTORS  = np.array([0.97, 1.00, 0.99, 1.04])   # Q1, Q2, Q3, Q4

country_order = ["US", "UK", "France", "Germany", "Japan"]


def seasonal_index(total_periods=TOTAL_PERIODS):
    """s[t] for t = 1..total_periods (index 0 unused). All 1.0 when disabled."""
    s = np.ones(total_periods + 1)
    if not USE_SEASONALITY:
        return s
    f = np.asarray(SEASONAL_FACTORS, float)
    f = f / np.exp(np.mean(np.log(f)))      # geometric mean 1 -> no drift
    for t in range(1, total_periods + 1):
        s[t] = f[(t - 1) % 4]
    return s


SEASONAL = seasonal_index()


def build_growth_series(anchor_periods, anchor_rates, total_periods=TOTAL_PERIODS):
    """
    anchor_periods: 1D array of anchor periods (e.g. [2,5,9,12,...,40])
    anchor_rates:   same length array of growth rates (decimals) at those anchors
    Returns growth[t] for t = 1..TOTAL_PERIODS (index 0 unused).
    Between anchors the rate is linearly interpolated; after the last anchor it
    is held flat (that flat tail is the top of the S).
    """
    anchor_periods = np.asarray(anchor_periods, int)
    anchor_rates   = np.asarray(anchor_rates, float)

    g = np.zeros(total_periods + 1)  # index 0 unused

    first_anchor = anchor_periods[0]
    last_anchor  = anchor_periods[-1]
    for t in range(first_anchor, total_periods + 1):
        if t <= first_anchor:
            g[t] = anchor_rates[0]
        elif t >= last_anchor:
            g[t] = anchor_rates[-1]
        else:
            idx = np.searchsorted(anchor_periods, t)
            if idx < len(anchor_periods) and anchor_periods[idx] == t:
                g[t] = anchor_rates[idx]
            else:
                left_p  = anchor_periods[idx - 1]
                right_p = anchor_periods[idx]
                left_g  = anchor_rates[idx - 1]
                right_g = anchor_rates[idx]
                alpha = (t - left_p) / (right_p - left_p)
                g[t] = left_g + alpha * (right_g - left_g)
    return g


def simulate_maus(start_mau, start_period, g, total_periods=TOTAL_PERIODS):
    """
    Underlying (deseasonalized) MAU path:
      - Periods < start_period: MAU = 0
      - Period start_period:    MAU = start_mau
      - Periods > start_period: MAU[t] = MAU[t-1] * (1 + g[t])
    """
    mau = np.zeros(total_periods + 1)
    for t in range(1, start_period):
        mau[t] = 0.0
    mau[start_period] = start_mau
    for t in range(start_period + 1, total_periods + 1):
        mau[t] = mau[t - 1] * (1 + g[t])
    return mau


def reported_maus(mau_trend):
    """Trend MAU with the optional seasonal wobble applied."""
    return mau_trend * SEASONAL


def monotonic_penalty(log_rates, lam=MONO_LAMBDA):
    """Penalize any INCREASE in anchor growth rates (we want a decaying rate)."""
    if lam <= 0:
        return np.zeros(max(len(log_rates) - 1, 0), dtype=float)
    log_rates = np.asarray(log_rates, float)
    violations = np.clip(log_rates[1:] - log_rates[:-1], 0, np.inf)
    return np.sqrt(lam) * violations


def smoothness_penalty(log_rates, lam=SMOOTH_LAMBDA):
    """
    Penalize curvature of the LOG growth rate (second differences).

    Penalizing curvature of the raw rate would force a straight-line decay,
    which cannot both hold the Year-3 target and land on the Year-10 ceiling —
    the solver answers that by crashing the rate and re-accelerating, which is
    a double hump, not an S. In log space a constant-percentage decay
    (g_t = g_0 * d^t) is curvature-free, so the natural answer is a clean,
    monotone, ever-flattening S.
    """
    if lam <= 0 or len(log_rates) < 3:
        return np.zeros(max(len(log_rates) - 2, 0), dtype=float)
    log_rates = np.asarray(log_rates, float)
    second_diff = log_rates[2:] - 2 * log_rates[1:-1] + log_rates[:-2]
    return np.sqrt(lam) * second_diff


# ============================================================
# 2. SATURATION, DERIVED FROM THE UMG2 VIDEO P&L MODEL
#    (v135MK, sheet "MAUs - End of Year Targets")
#
#    That workbook builds MAUs as
#        MAU(year) = SOM(year) x %Aware(year) x %Converted
#    and only carries %Aware to 2032. It gets there in two stages:
#      * three logistic years:  aware_max / (1 + exp(-1.5 * (year - midpoint)))
#      * the year the logistic tops out, %Aware is pinned to aware_max, and
#        every year after that the workbook extrapolates with
#            aware[y] = aware[y-1] * (1 + decay * g[y-1])
#        where g is the year-over-year awareness growth and the decay factor
#        is (1 + RRI(1, g[y-3], g[y-2])) — i.e. the growth rate keeps decaying
#        at the rate it was already decaying, with a one-year lag.
#
#    Running that same recursion out to 2036 gives the 10-year saturation
#    level. The replication is checked against the workbook's own cells below.
# ============================================================

PL_SOM_2025 = {   # B128:C132, "SOM Population (M)"
    "US": 52.17, "UK": 14.31, "France": 7.92, "Germany": 10.8, "Japan": 12.41,
}
PL_AWARE_MAX = {  # B136:C140, "Awareness by Country"
    "US": 0.395, "UK": 0.2575, "France": 0.2171, "Germany": 0.239, "Japan": 0.2932,
}
PL_CONVERSION = { # E113:E117, "Conversion Rate - by Country"
    "US": 0.473981, "UK": 0.349249, "France": 0.473981,
    "Germany": 0.438695, "Japan": 0.438418,
}
PL_LOGISTIC_MID = {  # from the %Aware formulas in rows 21/31/41/51/61
    "US": 2028.5, "UK": 2029.5, "France": 2029.5, "Germany": 2029.5, "Japan": 2029.5,
}
PL_POP_GROWTH = {    # rows 81-85, columns D:O = 2026..2037
    "US":      [.004944, .004711, .004629, .004516, .004359, .004264,
                .004175, .004096, .004039, .003958, .003848, .003769],
    "Japan":   [-.00585, -.006115, -.006284, -.006387, -.006462, -.006525,
                -.006556, -.006562, -.00656, -.006564, -.006523, -.006464],
    "UK":      [.005348, .005011, .004785, .004461, .004271, .004059,
                .003795, .003706, .003553, .003394, .003261, .003071],
    "Germany": [-.005214, -.002418, -.002506, -.00259, -.002584, -.002641,
                -.002674, -.002659, -.002631, -.002638, -.002863, -.002916],
    "France":  [.001346, .001211, .001123, .001058, .001011, .000948,
                .000927, .000923, .000949, .000976, .000961, .000968],
}

PL_LAUNCH_YEAR = 2027          # US launch; the notebook's period 1 sits here
PL_FINAL_YEAR  = 2036          # 10th year of the plan

# Workbook values to check the replication against (row 25/35/45/55/65, 2032).
PL_CHECK_2032 = {
    "US": 13.49283, "UK": 1.624702, "France": 1.004448,
    "Germany": 1.356704, "Japan": 1.866592,
}


def pl_som(country):
    """SOM by year, grown at the workbook's population growth rates."""
    out = {2025: PL_SOM_2025[country]}
    for i, year in enumerate(range(2026, 2038)):
        out[year] = out[year - 1] * (1 + PL_POP_GROWTH[country][i])
    return out


def pl_awareness(country, final_year=PL_FINAL_YEAR):
    """%Aware by year: the workbook's logistic, then its own extrapolation."""
    a = {}
    mid = PL_LOGISTIC_MID[country]
    for year in (mid - 1.5, mid - 0.5, mid + 0.5):
        a[int(year)] = PL_AWARE_MAX[country] / (1 + np.exp(-1.5 * (year - mid)))
    top_year = int(mid + 1.5)
    a[top_year] = PL_AWARE_MAX[country]          # the workbook's XLOOKUP row
    for year in range(top_year + 1, final_year + 1):
        g_older = a[year - 3] / a[year - 4] - 1
        g_newer = a[year - 2] / a[year - 3] - 1
        decay   = g_newer / g_older              # = 1 + RRI(1, g_older, g_newer)
        a[year] = a[year - 1] * (1 + decay * (a[year - 1] / a[year - 2] - 1))
    return a


def pl_mau_path(country, final_year=PL_FINAL_YEAR):
    """The workbook's MAU by year, extended to final_year."""
    som, aware = pl_som(country), pl_awareness(country, final_year)
    return {y: som[y] * aware[y] * PL_CONVERSION[country]
            for y in sorted(aware) if y <= final_year}


def pl_year_three(country):
    """
    The workbook year this notebook's Year-3 (period 12) target lines up with.

    The US lines up on the calendar: notebook period 12 is the end of year 3,
    and its 8.574M target sits against the workbook's 2029 (6.759M, ~27% above).

    The other four are deliberately ahead of the workbook. Their targets at
    periods 8 and 12 track the workbook's 2029 and 2030 (UK 0.4487 vs 0.421 and
    0.9015 vs 0.895; France 0.2810 vs 0.263 and 0.5625 vs 0.5567; Germany
    0.3851 vs 0.359 and 0.7681 vs 0.7574), while with launch at period 5 those
    periods are calendar 2028 and 2029. That one-year lead is intentional — the
    non-US plan is more aggressive than the workbook — so the targets are taken
    at face value and mapped to 2030 here.
    """
    return 2029 if country == "US" else 2030


# --- verify the replication before anything depends on it -------------------
print("=== P&L model replication check (2032, M) ===")
for _c in country_order:
    _got, _want = pl_mau_path(_c)[2032], PL_CHECK_2032[_c]
    _flag = "OK" if abs(_got / _want - 1) < 1e-4 else "MISMATCH"
    print(f"  {_c:8} replicated={_got:8.6f}  workbook={_want:8.6f}  {_flag}")
    assert _flag == "OK", f"{_c}: replication of the P&L model does not match"

print(f"\n=== P&L model extended to {PL_FINAL_YEAR} (M) ===")
_years = list(range(PL_LAUNCH_YEAR, PL_FINAL_YEAR + 1))
print("          " + "".join(f"{y:>8}" for y in _years))
for _c in country_order:
    _p = pl_mau_path(_c)
    print(f"  {_c:8}" + "".join(f"{_p.get(y, 0.0):8.2f}" for y in _years))


# ============================================================
# 3. COUNTRY CONFIGS
#    - Y1-Y3 targets (periods 4, 8, 12) are UNCHANGED commitments.
#    - The Year-10 level is no longer assumed: it comes from the P&L model
#      replicated in section 2, via CEILING_BASIS.
# ============================================================

# Anchors: dense through the launch/ramp years, sparse across the plateau.
US_ANCHORS    = np.array([2, 5, 9, 12, 16, 20, 24, 28, 32, 36, 40], dtype=int)
LATER_ANCHORS = np.array([5, 9, 12, 16, 20, 24, 28, 32, 36, 40], dtype=int)

country_configs = {
    "US": {
        "start_period": 1,
        "anchor_periods": US_ANCHORS,
        "target_periods": np.array([4, 8, 12], dtype=int),
        "target_maus": np.array([
            0.7749768,
            4.2678931,
            8.5744581
        ], dtype=float),
        "x0_start_mau": 0.5,
        "x0_first_g": 0.60,
    },

    "Japan": {
        "start_period": 5,
        "anchor_periods": LATER_ANCHORS,
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.6226316,
            1.2373884
        ], dtype=float),
        "x0_start_mau": 0.03,
        "x0_first_g": 0.37,
    },

    "UK": {
        "start_period": 5,
        "anchor_periods": LATER_ANCHORS,
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.4487472,
            0.9015061
        ], dtype=float),
        "x0_start_mau": 0.05,
        "x0_first_g": 0.37,
    },

    "France": {
        "start_period": 5,
        "anchor_periods": LATER_ANCHORS,
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.2809596,
            0.5625267
        ], dtype=float),
        "x0_start_mau": 0.03,
        "x0_first_g": 0.37,
    },

    "Germany": {
        "start_period": 5,
        "anchor_periods": LATER_ANCHORS,
        "target_periods": np.array([8, 12], dtype=int),
        "target_maus": np.array([
            0.3850503,
            0.7681255
        ], dtype=float),
        "x0_start_mau": 0.03,
        "x0_first_g": 0.37,
    },
}


def terminal_mau(country, basis=None):
    """
    The Year-10 (period 40) MAU level, derived from the P&L model.

    "absolute" takes the workbook's own 2036 figure. "multiple" keeps this
    notebook's committed Year-3 target and applies the workbook's terminal
    multiple (2036 / its own Year-3), which is ~2.0-2.2x for every country.
    """
    basis = CEILING_BASIS if basis is None else basis
    path = pl_mau_path(country)
    pl_final = path[PL_FINAL_YEAR]
    if basis == "absolute":
        return pl_final
    if basis == "multiple":
        pl_y3 = path[pl_year_three(country)]
        notebook_y3 = country_configs[country]["target_maus"][-1]
        return notebook_y3 * (pl_final / pl_y3)
    raise ValueError(f"CEILING_BASIS must be 'absolute' or 'multiple', got {basis!r}")


def initial_guess(cfg):
    """
    Unknowns are solved in LOG space: x = [log(start_mau), log(anchor_rates)...].
    That keeps MAU and every growth rate strictly positive without bounds, and
    makes a constant-percentage decay the zero-penalty default.
    """
    n = len(cfg["anchor_periods"])
    g_hi = cfg["x0_first_g"]
    g_lo = TERMINAL_GROWTH
    ramp = g_hi * (g_lo / g_hi) ** (np.arange(n) / max(n - 1, 1))
    return np.log(np.concatenate([[cfg["x0_start_mau"]], ramp]))


# ============================================================
# 4. SOLVER PER COUNTRY (start_mau + anchors, S-curve constrained)
# ============================================================

def solve_country(name, cfg, basis=None):
    start_period   = cfg["start_period"]
    anchor_periods = cfg["anchor_periods"]
    target_periods = cfg["target_periods"]
    target_maus    = cfg["target_maus"]
    x0             = initial_guess(cfg)

    terminal = terminal_mau(name, basis)

    # x = [log(start_mau), log(anchor_rates)...]
    def residuals(x):
        start_mau    = np.exp(x[0])
        log_rates    = x[1:]
        anchor_rates = np.exp(log_rates)

        g    = build_growth_series(anchor_periods, anchor_rates)
        mau  = reported_maus(simulate_maus(start_mau, start_period, g))

        # Relative errors so the 0.77M target counts as much as the 41M one.
        tgt_resids  = W_TARGET  * (mau[target_periods] / target_maus - 1.0)
        ceil_resid  = W_CEILING * np.array([mau[TOTAL_PERIODS] / terminal - 1.0])
        term_resid  = W_TERMINAL_G * np.array([
            np.log(anchor_rates[-1] / TERMINAL_GROWTH)
        ])

        mono_resids   = monotonic_penalty(log_rates)
        smooth_resids = smoothness_penalty(log_rates)

        return np.concatenate([tgt_resids, ceil_resid, term_resid,
                               mono_resids, smooth_resids])

    # Bounds in log space: MAU free and positive, every rate <= MAX_GROWTH.
    lower_bounds = np.full_like(x0, -np.inf)
    upper_bounds = np.concatenate([[np.inf],
                                   np.full(len(x0) - 1, np.log(MAX_GROWTH))])
    x0 = np.minimum(x0, upper_bounds)

    result = least_squares(
        residuals,
        x0,
        bounds=(lower_bounds, upper_bounds),
        max_nfev=200000,
        xtol=1e-14,
        ftol=1e-14,
        gtol=1e-14,
    )
    x_opt        = result.x
    start_mau    = np.exp(x_opt[0])
    anchor_rates = np.exp(x_opt[1:])

    g_full     = build_growth_series(anchor_periods, anchor_rates)
    mau_trend  = simulate_maus(start_mau, start_period, g_full)
    mau_report = reported_maus(mau_trend)

    return {
        "start_mau":     start_mau,
        "anchor_rates":  anchor_rates,
        "growth_series": g_full,       # decimals per period
        "mau_series":    mau_report,   # MAUs per period (M), seasonality applied
        "mau_trend":     mau_trend,    # MAUs per period (M), trend only
        "terminal":      terminal,     # Year-10 level implied by the P&L model
        "cost":          result.cost,
    }


# ============================================================
# 5. RUN SOLVER FOR ALL COUNTRIES
# ============================================================

results = {}

for country_name in country_order:
    cfg = country_configs[country_name]
    print(f"\n=== Solving for {country_name} ===")
    res = solve_country(country_name, cfg)
    results[country_name] = res

    print(f"  Start MAU (Period {cfg['start_period']}): {res['start_mau']:.6f} M")
    mau = res["mau_series"]
    for tp, target in zip(cfg["target_periods"], cfg["target_maus"]):
        err = mau[tp] / target - 1.0
        print(f"    Period {tp:>2}: simulated={mau[tp]:.6f}  target={target:.6f}  err={err*100:+.4f}%")
    print(f"    Period 40: simulated={mau[TOTAL_PERIODS]:.4f} M  "
          f"target={res['terminal']:.4f} M  "
          f"(P&L-derived, basis={CEILING_BASIS!r})")
    print("  Anchor growth rates (%):")
    print("    " + "  ".join(
        f"P{p}:{r*100:.2f}" for p, r in zip(cfg["anchor_periods"], res["anchor_rates"])
    ))


# ============================================================
# 6. SENSITIVITY: HOW FAR DOES CEILING_BASIS MOVE THE PATH?
#    Years 1-3 are pinned by the targets either way, so the two bases can only
#    differ from Year 4 on. This prints the worst-case gap so the size of the
#    choice is visible rather than implied.
# ============================================================

def basis_comparison():
    both = {}
    for basis in ("multiple", "absolute"):
        both[basis] = {c: solve_country(c, country_configs[c], basis)["mau_series"]
                       for c in country_order}
    rows = []
    for c in country_order:
        m, a = both["multiple"][c], both["absolute"][c]
        y10 = (m[TOTAL_PERIODS], a[TOTAL_PERIODS])
        gaps = [abs(m[t] / a[t] - 1) for t in range(13, TOTAL_PERIODS + 1) if a[t] > 0]
        rows.append((c, y10[0], y10[1], y10[0] / y10[1] - 1, max(gaps)))
    return pd.DataFrame(rows, columns=["Country", "Y10_multiple", "Y10_absolute",
                                       "Y10_diff", "max_diff_Y4_on"])


_sens = basis_comparison()
print("\n=== CEILING_BASIS sensitivity (Years 1-3 identical under both) ===")
for _, r in _sens.iterrows():
    print(f"  {r.Country:8} Y10: {r.Y10_multiple:6.2f} M (multiple) vs "
          f"{r.Y10_absolute:6.2f} M (absolute)   "
          f"Y10 gap {r.Y10_diff:+6.1%}   worst gap from Y4 on {r.max_diff_Y4_on:6.1%}")


# ============================================================
# 6. YEARLY SUMMARY (Year = 4 periods)
# ============================================================

years = np.arange(1, TOTAL_PERIODS // 4 + 1)
summary = pd.DataFrame({"Year": years, "Period": years * 4})
for country_name in country_order:
    m = results[country_name]["mau_series"]
    summary[country_name] = [round(m[y * 4], 4) for y in years]
print("\n=== Exit-quarter MAUs by year (millions) ===")
print(summary.to_string(index=False))


# ============================================================
# 7. BUILD EXCEL OUTPUT
# ============================================================

def report_from(country_name):
    """First period a country reports growth (period after launch)."""
    return country_configs[country_name]["start_period"] + 1


# Sheet 1: Growth percentages per period
growth_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    g = results[country_name]["growth_series"]
    first = report_from(country_name)
    growth_df[country_name + "_growth_pct"] = [
        g[t] * 100.0 if t >= first else np.nan for t in PERIODS
    ]

# Sheet 2: MAUs per period — ABSOLUTE NUMBERS WITH FULL DECIMALS
mau_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    m = results[country_name]["mau_series"]
    mau_df[country_name + "_MAU_absolute"] = [m[t] * 1_000_000.0 for t in PERIODS]

# Sheet 3: Starting Acquisition Rates per period — DECIMALS
acq_df = pd.DataFrame({"Period": PERIODS})
for country_name in country_order:
    g = results[country_name]["growth_series"]
    first = report_from(country_name)
    acq_df[country_name + "_acq_rate"] = [
        g[t] if t >= first else np.nan for t in PERIODS
    ]

output_filename = "growth_model_s_curve_P40_US_UK_FR_DE_JP.xlsx"
with pd.ExcelWriter(output_filename, engine="xlsxwriter") as writer:
    growth_df.to_excel(writer, sheet_name="Growth_Percentages", index=False)
    mau_df.to_excel(writer, sheet_name="MAUs_Absolute", index=False)
    acq_df.to_excel(writer, sheet_name="Starting_Acq_Rates", index=False)
    summary.to_excel(writer, sheet_name="Yearly_Summary", index=False)
    _sens.to_excel(writer, sheet_name="Basis_Sensitivity", index=False)

print(f"\nExcel file written to: {output_filename}")
